In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import numpy as np 
import PIL.Image as Image
import pandas as pd
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
import os
import matplotlib.pyplot as plt


c:\pc data\DATA SCIENCE AND ML\PYTHON\deep learning\Udemy - Complete Tensorflow 2 and Keras Deep Learning Bootcamp 2022-6\code place\envoo\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [6]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
class DatasetMaker(Dataset):
    def __init__(self,path,transform=None):
        super().__init__()
        self.map=pd.read_csv(path,index_col=0)
        self.transform=transform
    def __len__(self):
        return len(self.map)
    def __getitem__(self,idx):
        y=self.map['encoded_label'][idx]
        img=Image.open(self.map['ImagePath'][idx]).convert('L')
        if self.transform:
            img=self.transform(img)
        return img,y
        

In [3]:
#temp train data object just to calculate the mean and std that we will use in the main train data object and for the test and validation data too.
temp_train_transform=transforms.Compose([transforms.CenterCrop((480,480)),transforms.Resize((224,224)),transforms.ToTensor()])
train_path=r"DataSet\train_data.csv"
temp_train_Dataset=DatasetMaker(train_path,transform=temp_train_transform)
all=[temp_train_Dataset[i][0] for i in range(len(temp_train_Dataset))]
a=torch.stack(all,dim=0)
train_mean=a[:,0,:,:].mean()
train_std=a[:,0,:,:].std()

In [4]:
# we apply plausible (Physically) Augmentations to the train data only
# degrees=10 (rotation), translate=(0.05, 0.05) (horizontal/vertical shift)
# scale=(0.98, 1.02) (zoom in/out slightly)
train_transform=transforms.Compose([transforms.CenterCrop((480,480)),
                                    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.98, 1.02)),                        
                                    transforms.Resize((224,224)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(mean=train_mean,std=train_std)])

val_transform=transforms.Compose([transforms.CenterCrop((480,480)),
                                    transforms.Resize((224,224)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(mean=train_mean,std=train_std)])


test_transform=transforms.Compose([transforms.CenterCrop((480,480)),
                                    transforms.Resize((224,224)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(mean=train_mean,std=train_std)])

In [5]:
train_ds=DatasetMaker(path=r"DataSet\train_data.csv",transform=train_transform)
test_ds=DatasetMaker(path=r"DataSet\test_data.csv",transform=test_transform)
val_ds=DatasetMaker(path=r"DataSet\val_data.csv",transform=test_transform)

In [29]:
class CreateModel(nn.Module):
    def __init__(self,num_classes=2000):
        super().__init__()
        #input images are batch*1*224*224
        #first conv block
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)) # here we will have output of batch*64*112*112 max pool have default stride of kernel size which is 2 here.
        
        #second conv block
        self.block2 = nn.Sequential(nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)) # the ouput will be batch*128*56*56
        
        #3rd conv block 
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)) # the output will be batch*256*28*28
        
        # 4th conv block 
        self.block4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)) # the output here will be batch*512*14*14
        
        #global average pooling layer to reduce the spatial dimensions to 1x1 so the dimension doesnt explode
        self.gap = nn.AdaptiveAvgPool2d((1, 1)) # the output will be batch*512*1*1 (it takes the average of each filter output)
        
        # fully connected block 
        self.fc1 = nn.Sequential(nn.Flatten(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features=512*1*1,out_features=1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(in_features=1024,out_features=num_classes)) # this will output batch*2000 for the softmax function to use.      
        
        # define model behavior
    def forward(x):
        x=self.block1(x)
        x=self.block2(x)
        x=self.block3(x)
        x=self.block4(x)
        x=self.gap(x)
        x=self.fc1(x)
        return x

In [30]:
myCNN=CreateModel().to(device)

In [31]:
num_of_parameters=sum(p.numel() for p in myCNN.parameters())
print('the model parameters count is :',num_of_parameters)

the model parameters count is : 4127056


In [32]:
optimizer=torch.optim.Adam(myCNN.parameters(),lr=3e-4,weight_decay=1e-4)